# Community Service — Dataset Documentation

**Dataset:** `community_service.csv`

**Purpose:** Tracks student community service activities, reported service hours, and verification statuses within the synthetic Graduation Tracker student information system. Supports monitoring community service completion, calculating verified service hours, and identifying students with outstanding graduation requirements.

**Record level:** One row per student community service activity.

**Primary key:** `service_id`

**Foreign key:** `student_id` → `students.csv`

**Fields:**

- `service_id`: Unique community service record identifier using the format `SVC00001`.
- `student_id`: Student identifier linking community service records to the student dataset.
- `service_date`: Date of the community service activity, formatted as `YYYY-MM-DD`.
- `hours_reported`: Number of community service hours reported for the activity.
- `verification_status`: Verification outcome of the reported service hours (Verified, Pending, or Rejected).

**Business rules:**

- Each student receives between 0 and 4 randomly generated community service records.
- Students may have no community service records.
- Service dates are randomly generated between 2023 and 2026.
- All service activities are assigned to the 15th day of the randomly selected month.
- Reported service hours are randomly selected from the following values: 1, 2, 3, 4, 5, or 8 hours.
- Verification statuses follow these probability distributions:
  - Verified: 75%
  - Pending: 20%
  - Rejected: 5%
- Only records with a Verified status contribute to a student's total verified community service hours.
- Pending and Rejected records do not contribute to verified service hour totals.
- Verified service hours are calculated by summing `hours_reported` for each student with Verified records.
- Service IDs must be unique.
- Every `student_id` must exist in the student dataset.
- All reported service hours must be greater than zero.
- Random generation uses a fixed seed of 42 to support reproducibility.

**Data validation:**

- Verify all service IDs are unique.
- Verify all student IDs exist in the student dataset.
- Verify all reported service hours are greater than zero.

**Data source:** `students.csv` provides the student identifiers used to generate community service activity records.

**Generation method:** Synthetic community service records generated using Python, pandas, and NumPy random sampling.

**Derived metric:** `verified_hours` represents the total approved community service hours for each student. It is calculated from records with a Verified status and is not included as a separate field in the exported `community_service.csv` dataset.

**Limitations:** Community service activities, dates, reported hours, and verification statuses are synthetically generated and do not represent actual student service records. The dataset does not identify service organizations, activity descriptions, supervising personnel, or supporting documentation. Service dates are not validated against student enrollment dates or graduation cohorts. The notebook does not define a required number of community service hours for graduation or calculate whether students have fulfilled that requirement. Students without verified service records are excluded from the derived `verified_hours` summary unless explicitly added during subsequent data processing.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import drive

In [ ]:
# Connect to Google Drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Create Folder Path

folder_path = Path('/content/drive/MyDrive/Graduation_Tracker/data/ground_truth')

In [ ]:
# Set random seed

rng = np.random.default_rng(42)

In [ ]:
# Generate Community Service Hours records

students = pd.read_csv(folder_path/'students.csv')

records = []

for id in students.student_id:
  for r in range(int(rng.integers(0,5))):
    records.append({
        'student_id':id,
        'service_date':f'{int(rng.integers(2023, 2027))}-{int(rng.integers(1,13)):02d}-15',
        'hours_reported':int(rng.choice([1, 2, 3, 4, 5, 8])),
        'verification_status':rng.choice(['Verified', 'Pending', 'Rejected'], p = [0.75, 0.20, 0.05])
    })

cs_hours = pd.DataFrame(records)

cs_hours.insert(0, 'service_id', [f'SVC{i:05d}' for i in range(1, len(cs_hours) + 1)])

display(cs_hours.head())

,service_id,student_id,service_date,hours_reported,verification_status
0,SVC00001,1145269,2023-07-15,8,Verified
1,SVC00002,1145270,2026-08-15,3,Verified
2,SVC00003,1145270,2023-12-15,2,Pending
3,SVC00004,1145272,2026-04-15,8,Verified
4,SVC00005,1145273,2023-10-15,3,Verified


In [ ]:
# Validate dataset

assert cs_hours.service_id.is_unique

assert cs_hours.student_id.isin(students.student_id).all()

assert cs_hours.hours_reported.gt(0).all()



In [ ]:
# Calculate verified hours

verified_hours = (cs_hours.loc[cs_hours.verification_status.eq('Verified')]
                  .groupby('student_id', as_index = False).hours_reported.sum())

verified_hours.rename(columns = {'hours_reported':'verified_hours'}, inplace = True)

display(verified_hours.head())

,student_id,verified_hours
0,1145269,8
1,1145270,3
2,1145272,8
3,1145273,7
4,1145275,23


In [ ]:
# Save CSV to Google Drive

csv_path = folder_path / 'community_service.csv'

cs_hours.to_csv(csv_path, index = False)